# 06 — Pooled walk-forward validation

This notebook re-fits quantile targets, K-means candle clusters, numerical scaling and categorical encoding independently within each expanding training window. It does **not** use the globally fitted labels or encoders from Notebook 03. Do not deploy models based on these exploratory scores alone.


## 1. Load saved causal features
Run Notebook 03 first. The master table retains the raw causal numeric and calendar features. We recompute `direction` and `candle_type` in each fold.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.walk_forward import expanding_folds, fold_data, evaluate_fold, diagnose_latest_z
from stock_ml_dynamic.master_table import fetch_universe_prices
from stock_ml_dynamic.db import engine

INTERVAL = "1d"
MASTER_DIR = Path("artifacts/master_tables") / INTERVAL
master, latest, _, manifest = load_master(MASTER_DIR)
all_features = pd.concat([master, latest], ignore_index=True)
print(f"Historical rows: {len(master):,}; latest rows: {len(latest):,}; symbols: {all_features.symbol.nunique()}")


## 2. Diagnose VYLR adjusted z-score
The previous audit found one missing latest `adj_z`. This cell reads raw PostgreSQL prices, checks the preceding 30 bars, and reports the cause without imputing values.


In [ ]:
with engine().connect() as conn:
    vylr_prices = fetch_universe_prices(conn, interval=INTERVAL, bars_per_symbol=80, symbols=["VYLR"])
print(diagnose_latest_z(vylr_prices, symbol="VYLR"))
display(vylr_prices[["bar_timestamp", "adj_close_price"]].tail(35))


## 3. Configure expanding folds
All tickers share global UTC boundaries. Training labels must have realized by the training cutoff; test features begin after that cutoff. Each test label must be realized by the test-end timestamp.


In [ ]:
N_FOLDS = 3
INITIAL_TRAIN_FRACTION = 0.55
SELECTED_K = 4  # Use a fixed cluster count for cross-fold comparability.
CLUSTER_FIT_LIMIT = 30000  # Reproducible training-only subsample for K-means.
folds = expanding_folds(all_features, n_folds=N_FOLDS, initial_fraction=INITIAL_TRAIN_FRACTION)
display(pd.DataFrame([vars(f) for f in folds]))


## 4. Train and evaluate within each fold
This may take several minutes on the complete 220-stock dataset. Models are trained only inside this notebook; no Streamlit training is performed.


In [ ]:
reports = []
for fold in folds:
    print(f"Fold {fold.number}: train through {fold.train_end}, test through {fold.test_end}", flush=True)
    tr, te, X_tr, X_te, transforms = fold_data(all_features, fold, selected_k=SELECTED_K, cluster_fit_limit=CLUSTER_FIT_LIMIT)
    score = evaluate_fold(tr, te, X_tr, X_te, fold_number=fold.number)
    score["train_end"] = str(fold.train_end)
    score["test_end"] = str(fold.test_end)
    reports.append(score)
    display(score.round(5))
results = pd.concat(reports, ignore_index=True)


## 5. Compare model stability across folds
A classifier should beat its baseline consistently, not only in one window. Lower regression MAE is better.


In [ ]:
display(results.groupby(["task", "model"], as_index=False).agg({"accuracy":"mean", "balanced_accuracy":"mean", "return_mae":"mean", "return_rmse":"mean", "return_r2":"mean", "price_mae":"mean"}).round(5))
fig, axes = plt.subplots(1,2,figsize=(13,4))
for name, sub in results.query("task == 'classification'").groupby("model"):
    axes[0].plot(sub.fold,sub.balanced_accuracy,marker="o",label=name)
axes[0].set(title="Classification balanced accuracy",xlabel="Fold",ylabel="Balanced accuracy")
axes[0].legend()
for name, sub in results.query("task == 'regression'").groupby("model"):
    axes[1].plot(sub.fold,sub.return_mae,marker="o",label=name)
axes[1].set(title="Regression next-return MAE",xlabel="Fold",ylabel="MAE")
axes[1].legend()
plt.tight_layout()
plt.show()


## 6. Save evaluation reports (not production models)
Model promotion should be a separate decision after a genuinely untouched holdout.


In [ ]:
out = Path("artifacts/walk_forward") / INTERVAL
out.mkdir(parents=True, exist_ok=True)
results.to_csv(out / "fold_metrics.csv", index=False)
results.to_parquet(out / "fold_metrics.parquet", index=False)
print("Saved:", out.resolve())
